In [3]:
import torch
import segmentation_models_pytorch as smp
from torchinfo import summary

def inspect_models():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # 1. Define the models exactly as you have them in your pipeline
    models = {
        "DeepLabV3": smp.DeepLabV3(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1),
        "UnetPlusPlus": smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1),
        "Segformer": smp.Segformer(encoder_name="mit_b0", encoder_weights=None, in_channels=3, classes=1),
    }

    # 2. Define the simulated input size you want to test
    # Format: (Batch Size, Channels, Height, Width)
    # Let's use the 256x256 size you used during training. 
    # You can change this to (1, 3, 500, 500) to test your ONNX export dimensions!
    TEST_INPUT_SIZE = (1, 3, 512, 512)

    for name, model in models.items():
        model = model.to(device)
        print("=" * 80)
        print(f"MODEL: {name}")
        print("=" * 80)
        
        # 3. Generate the summary
        # This passes a dummy tensor of your exact size through the model and maps every layer
        model_stats = summary(
            model, 
            input_size=TEST_INPUT_SIZE,
            col_names=["input_size", "output_size", "num_params", "mult_adds"],
            col_width=18,
            row_settings=["var_names"],
            device=device,
            verbose=0 # Suppress automatic printing so we can format it cleanly
        )
        
        print(model_stats)
        print("\n")
        
        # Free up memory before loading the next model
        del model
        torch.cuda.empty_cache()

if __name__ == "__main__":
    inspect_models()

Using device: cuda

MODEL: DeepLabV3
Layer (type (var_name))                            Input Shape        Output Shape       Param #            Mult-Adds
DeepLabV3 (DeepLabV3)                              [1, 3, 512, 512]   [1, 1, 512, 512]   --                 --
├─ResNetEncoder (encoder)                          [1, 3, 512, 512]   [1, 3, 512, 512]   --                 --
│    └─Conv2d (conv1)                              [1, 3, 512, 512]   [1, 64, 256, 256]  9,408              616,562,688
│    └─BatchNorm2d (bn1)                           [1, 64, 256, 256]  [1, 64, 256, 256]  128                128
│    └─ReLU (relu)                                 [1, 64, 256, 256]  [1, 64, 256, 256]  --                 --
│    └─MaxPool2d (maxpool)                         [1, 64, 256, 256]  [1, 64, 128, 128]  --                 --
│    └─Sequential (layer1)                         [1, 64, 128, 128]  [1, 64, 128, 128]  --                 --
│    │    └─BasicBlock (0)                         [1, 64,